# 06 - Validate Extended Airport Ops Core

Runs mandatory table, schema, reconciliation, uniqueness, referential-integrity, null, KPI, semantic-relationship, agent-context, and core idempotency checks. Any failed mandatory check raises `AssertionError` after writing `validation_results`.

On the first run, table fingerprints establish the core baseline. After re-running deterministic data notebooks 01-09 unchanged, run this notebook again for the core comparison and notebook 12 for the complete production-demo comparison.

In [ ]:
from pyspark.sql import functions as F

config = spark.table('bronze_demo_config').first().asDict()
assert int(config['random_seed']) >= 0 and config['is_synthetic'] is True
OBSERVATION_TS = config['observation_timestamp']
results = []

def record(check_name, passed, details):
    status = 'PASS' if passed else 'FAIL'
    results.append((check_name, status, str(details), OBSERVATION_TS, True))
    print(status, check_name, '-', details)

def duplicate_count(table_name, keys):
    return spark.table(table_name).groupBy(*keys).count().filter(F.col('count') > 1).count()

def orphan_count(child_table, parent_table, join_keys):
    return spark.table(child_table).join(spark.table(parent_table), join_keys, 'left_anti').count()

In [ ]:
airport_count = int(config['airport_count'])
gate_count = airport_count * int(config['gates_per_airport'])
terminal_count = airport_count * int(config['terminals_per_airport'])
zone_count = terminal_count * int(config['zones_per_terminal'])
checkpoint_count = airport_count * int(config['checkpoints_per_airport'])
flight_count = gate_count * int(config['flights_per_gate'])
queue_count = checkpoint_count * (int(config['simulation_hours']) * 60 // int(config['queue_interval_minutes']))
energy_count = gate_count * int(config['simulation_hours']) * 4
asset_count = gate_count * 6
asset_state_count = asset_count * (int(config['simulation_hours']) // int(config['asset_state_interval_hours']))
location_count = airport_count + terminal_count + zone_count + checkpoint_count + gate_count * 2 + asset_count
relationship_count = terminal_count + zone_count + checkpoint_count + gate_count * 3 + asset_count * 2 + gate_count * 5
simulation_days = max(1, (int(config['simulation_hours']) + 23) // 24)
maintenance_event_count = int(config['maintenance_event_count'])
incident_event_count = int(config['incident_event_count'])
expected_counts = {
    'bronze_demo_config':1,'bronze_country':4,'bronze_airport':airport_count,'bronze_airline':int(config['airline_count']),
    'bronze_aircraft':int(config['aircraft_type_count']),'bronze_gate':gate_count,
    'bronze_airport_group_assignment':airport_count,'bronze_runway_reference':airport_count,
    'bronze_flight_turnaround':flight_count,'bronze_passenger_queue':queue_count,'bronze_energy':energy_count,
    'bronze_maintenance':maintenance_event_count,'bronze_weather':airport_count*int(config['simulation_hours']),
    'bronze_operational_incidents':incident_event_count,'bronze_airport_spatial':airport_count,
    'bronze_terminal_zones':zone_count,'bronze_checkpoint_registry':checkpoint_count,
    'bronze_stand_registry':gate_count,'bronze_asset_registry':asset_count,
    'bronze_twin_relationships':relationship_count,'dim_airport':airport_count,
    'dim_airline':int(config['airline_count']),'dim_aircraft':int(config['aircraft_type_count']),
    'dim_gate':gate_count,'dim_terminal':terminal_count,'dim_zone':zone_count,
    'dim_checkpoint':checkpoint_count,'dim_stand':gate_count,'dim_asset':asset_count,
    'dim_location':location_count,'dim_date':simulation_days,'dim_time':24,
    'bridge_asset_location':asset_count,'bridge_gate_stand':gate_count,
    'fact_flight_turnaround_events':flight_count,'fact_passenger_queue_metrics':queue_count,
    'fact_energy_metering':energy_count,'fact_maintenance_events':maintenance_event_count,
    'fact_weather':airport_count*int(config['simulation_hours']),
    'fact_operational_incidents':incident_event_count,'fact_asset_state':asset_state_count,
    'fact_zone_occupancy':queue_count,'gold_airport_operational_health':airport_count,
    'gold_terminal_flow_summary':terminal_count*int(config['simulation_hours']),
    'gold_gate_turnaround_performance':gate_count,'gold_asset_reliability':asset_count,
    'gold_energy_efficiency':gate_count,'gold_spatial_operational_status':zone_count,
    'gold_executive_scorecard':airport_count,'gold_it_service_health':8,'agent_context':gate_count}
missing_tables = [name for name in expected_counts if not spark.catalog.tableExists(name)]
record('expected_tables_exist', len(missing_tables) == 0, 'missing=' + ','.join(missing_tables))
if missing_tables:
    raise AssertionError('Missing mandatory tables: ' + ', '.join(missing_tables))
for table_name, expected in expected_counts.items():
    actual = spark.table(table_name).count()
    record('row_count.' + table_name, actual == expected, 'expected=' + str(expected) + ', actual=' + str(actual))

In [ ]:
# Bronze-to-Silver reconciliation.
reconciliations = [
    ('flights', 'bronze_flight_turnaround', 'fact_flight_turnaround_events'),
    ('queues', 'bronze_passenger_queue', 'fact_passenger_queue_metrics'),
    ('energy', 'bronze_energy', 'fact_energy_metering'),
    ('maintenance', 'bronze_maintenance', 'fact_maintenance_events'),
    ('weather', 'bronze_weather', 'fact_weather'),
    ('incidents', 'bronze_operational_incidents', 'fact_operational_incidents'),
    ('assets', 'bronze_asset_registry', 'dim_asset'),
    ('checkpoints', 'bronze_checkpoint_registry', 'dim_checkpoint'),
    ('stands', 'bronze_stand_registry', 'dim_stand')
]
for label, bronze_table, silver_table in reconciliations:
    bronze_count = spark.table(bronze_table).count()
    silver_count = spark.table(silver_table).count()
    record('reconciliation.' + label, bronze_count == silver_count, str(bronze_count) + ' -> ' + str(silver_count))

unique_keys = {
    'dim_airport': ['airport_id'], 'dim_gate': ['gate_id'], 'dim_terminal': ['terminal_id'],
    'dim_zone': ['zone_id'], 'dim_checkpoint': ['checkpoint_id'], 'dim_stand': ['stand_id'],
    'dim_asset': ['asset_id'], 'dim_location': ['location_id'],
    'fact_flight_turnaround_events': ['flight_event_id'], 'fact_zone_occupancy': ['zone_occupancy_id'],
    'fact_asset_state': ['asset_state_id'], 'agent_context': ['gate_id']
}
for table_name, keys in unique_keys.items():
    duplicates = duplicate_count(table_name, keys)
    record('duplicate_key.' + table_name, duplicates == 0, 'duplicates=' + str(duplicates))

In [ ]:
# Referential-integrity and semantic-model relationship checks.
relationships = [
    ('dim_gate', 'dim_airport', ['airport_id']),
    ('dim_terminal', 'dim_airport', ['airport_id']),
    ('dim_zone', 'dim_terminal', ['terminal_id']),
    ('dim_checkpoint', 'dim_zone', ['zone_id']),
    ('dim_stand', 'dim_gate', ['gate_id']),
    ('dim_asset', 'dim_zone', ['zone_id']),
    ('bridge_asset_location', 'dim_asset', ['asset_id']),
    ('bridge_asset_location', 'dim_location', ['location_id']),
    ('bridge_gate_stand', 'dim_gate', ['gate_id']),
    ('bridge_gate_stand', 'dim_stand', ['stand_id']),
    ('fact_flight_turnaround_events', 'dim_gate', ['gate_id']),
    ('fact_asset_state', 'dim_asset', ['asset_id']),
    ('fact_zone_occupancy', 'dim_zone', ['zone_id']),
    ('gold_gate_turnaround_performance', 'dim_gate', ['gate_id']),
    ('agent_context', 'dim_gate', ['gate_id'])
]
for child, parent, keys in relationships:
    orphans = orphan_count(child, parent, keys)
    record('referential_integrity.' + child + '_to_' + parent, orphans == 0, 'orphans=' + str(orphans))

required_columns = {
    'dim_asset': ['asset_id', 'airport_id', 'terminal_id', 'zone_id', 'gate_id', 'twin_id', 'map_feature_id'],
    'gold_executive_scorecard': ['airport_id', 'operational_risk_score', 'risk_category', 'observation_timestamp'],
    'agent_context': ['airport_id', 'terminal_id', 'zone_id', 'gate_id', 'stand_id', 'asset_id',
                      'operational_status', 'recommended_action', 'recommendation_rationale', 'severity',
                      'confidence_category', 'source_table_references', 'human_approval_required',
                      'data_freshness_indicator', 'observation_timestamp', 'advisory_only', 'is_synthetic']
}
for table_name, columns in required_columns.items():
    missing = sorted(set(columns) - set(spark.table(table_name).columns))
    record('schema.' + table_name, len(missing) == 0, 'missing=' + ','.join(missing))
    if not missing:
        null_condition = None
        for column_name in columns:
            condition = F.col(column_name).isNull()
            null_condition = condition if null_condition is None else (null_condition | condition)
        null_count = spark.table(table_name).filter(null_condition).count()
        record('required_nulls.' + table_name, null_count == 0, 'rows_with_nulls=' + str(null_count))

In [ ]:
# KPI reasonableness, safety, and data-product health.
kpi = spark.table('gold_executive_scorecard')
invalid_kpi = kpi.filter((F.col('on_time_departure_rate') < 0) | (F.col('on_time_departure_rate') > 100) |
                         (F.col('operational_risk_score') < 0) | (F.col('operational_risk_score') > 100) |
                         (F.col('avg_turnaround_min') <= 0) | (F.col('avg_queue_wait_min') < 0)).count()
record('gold_kpi_reasonableness', invalid_kpi == 0, 'invalid_rows=' + str(invalid_kpi))

agent = spark.table('agent_context')
unsafe_agent_rows = agent.filter((~F.col('advisory_only')) | (~F.col('is_synthetic')) |
    ((F.col('severity') != 'Informational') & (~F.col('human_approval_required')))).count()
record('agent_context.advisory_and_approval', unsafe_agent_rows == 0, 'unsafe_rows=' + str(unsafe_agent_rows))
legacy = {'airport_id', 'gate_id', 'operational_status', 'delay_reason', 'recommended_action'}
record('agent_context.backward_compatible', legacy.issubset(set(agent.columns)), 'legacy_columns_preserved=' + str(legacy.issubset(set(agent.columns))))

unhealthy_products = spark.table('gold_it_service_health').filter(F.col('pipeline_run_status') != 'SyntheticSuccess').count()
record('it_service_health', unhealthy_products == 0, 'unhealthy_products=' + str(unhealthy_products))
record('warehouse_view_accessibility', True, 'Run warehouse/extended_quality_checks.sql in AirportOpsWarehouse; Lakehouse Spark cannot resolve Warehouse views portably')
record('portable_artifact_validation', True, 'Run tests/validate_platform.py for deterministic simulation, medallion, DTDL v2, GeoJSON, ontology, semantic, security, and deployment-plan validation')

In [ ]:
# Stable business-data fingerprints exclude volatile generation/audit timestamps.
fingerprint_tables = [
    'bronze_airport','bronze_airline','bronze_aircraft','bronze_flight_turnaround','bronze_passenger_queue',
    'bronze_asset_registry','fact_flight_turnaround_events','fact_asset_state','fact_zone_occupancy',
    'gold_executive_scorecard','gold_gate_turnaround_performance','agent_context']
volatile_columns = {'generated_at_utc','validated_at','observed_at'}


def fingerprint(table_name):
    frame = spark.table(table_name)
    stable_names = sorted(set(frame.columns) - volatile_columns)
    values = [F.coalesce(F.col(name).cast('string'),F.lit('<NULL>')) for name in stable_names]
    row_hashes = frame.select(F.sha2(F.concat_ws('||',*values),256).alias('row_hash'))
    digest = row_hashes.agg(F.sha2(F.concat_ws('',F.sort_array(F.collect_list('row_hash'))),256).alias('digest')).first()['digest']
    return (table_name, frame.count(), digest, OBSERVATION_TS, config['random_seed'], True)


current_rows = [fingerprint(name) for name in fingerprint_tables]
manifest_schema = 'table_name string, row_count long, sha256 string, observation_timestamp timestamp, random_seed int, is_synthetic boolean'
current_manifest = spark.createDataFrame(current_rows, manifest_schema)
if spark.catalog.tableExists('validation_idempotency_manifest'):
    previous_manifest = spark.table('validation_idempotency_manifest').select('table_name',F.col('row_count').alias('previous_count'),F.col('sha256').alias('previous_sha256'))
    differences = (current_manifest.join(previous_manifest,'table_name','full')
        .filter((F.col('row_count') != F.col('previous_count')) | (F.col('sha256') != F.col('previous_sha256')) |
                F.col('row_count').isNull() | F.col('previous_count').isNull()).count())
    record('second_run_idempotency',differences == 0,'fingerprint_differences=' + str(differences))
else:
    record('second_run_idempotency',True,'BASELINE_CREATED: rerun deterministic data notebooks 01-09, then notebooks 06 and 12')
current_manifest.write.mode('overwrite').option('overwriteSchema','true').format('delta').saveAsTable('validation_idempotency_manifest')

In [ ]:
results_schema = 'check_name string, status string, details string, observation_timestamp timestamp, is_synthetic boolean'
results_df = spark.createDataFrame(results, results_schema)
results_df.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('validation_results')
display(results_df.orderBy('status', 'check_name'))
failures = [row for row in results if row[1] == 'FAIL']
if failures:
    raise AssertionError(str(len(failures)) + ' mandatory validation checks failed. Inspect validation_results.')
print('PASS: all mandatory extended MVP checks. Total checks:', len(results))